# Coffee17 Focused Frequency-Efficiency — Preflight Only

No Coffee17 training is performed here. This notebook validates the new FReCSA/FRSA candidates and measures runtime efficiency before any expensive screening.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, json

WORK = Path("/kaggle/working")
REPO = WORK / "coffee-bean-classification"
SCIENTIFIC_CODE_COMMIT = "643411b91cf753ced77e5fc16104674c3e58f806"
NOTEBOOK_RUNTIME_REV = "FOCUSED-EFFICIENCY-PREFLIGHT-V1-20261007"

print("NOTEBOOK_RUNTIME_REV:", NOTEBOOK_RUNTIME_REV)
print("SCIENTIFIC_CODE_COMMIT:", SCIENTIFIC_CODE_COMMIT)
print("TRAINING: DISABLED")

os.chdir(WORK)
if REPO.exists():
    shutil.rmtree(REPO)

subprocess.run(
    ["git", "clone", "https://github.com/ediprin/coffee-bean-classification.git", str(REPO)],
    check=True,
    cwd=WORK,
)
subprocess.run(
    ["git", "checkout", SCIENTIFIC_CODE_COMMIT],
    check=True,
    cwd=REPO,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "--no-deps"],
    check=True,
    cwd=WORK,
)

if not __import__("torch").cuda.is_available():
    raise RuntimeError("Aktifkan Kaggle GPU untuk runtime-efficiency preflight.")

print("CUDA:", __import__("torch").cuda.get_device_name(0))


In [ ]:
# Unit tests for the exact candidate implementations.
subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q",
        "tests/test_frequency_screening.py",
    ],
    check=True,
    cwd=REPO,
)
print("FREQUENCY TESTS: PASS")


In [ ]:
# Real forward/backward + batch-1 latency/throughput/memory profiling.
OUT = WORK / "focused_efficiency_preflight.json"
cmd = [
    sys.executable, "-m",
    "bilinear_lmmd.experiments.profile_frequency_efficiency_v1",
    "--config", "configs/frequency_screening/FOCUSED_EFFICIENCY_V1.yaml",
    "--candidates", "B0", "H384", "C1", "C2", "C3", "C4",
    "--device", "cuda",
    "--warmup", "30",
    "--iterations", "100",
    "--output", str(OUT),
]
subprocess.run(cmd, check=True, cwd=REPO)

payload = json.loads(OUT.read_text())
print("\n=== COMPACT SUMMARY ===")
for row in payload["results"]:
    print(
        row["candidate"],
        "params=", row["parameter_count"],
        "MACs(G)=", round(row["conv_linear_macs_g"], 4),
        "median_ms=", round(row["latency_batch1_median_ms"], 3),
        "p95_ms=", round(row["latency_batch1_p95_ms"], 3),
        "img/s=", round(row["throughput_batch1_img_s"], 2),
        "peakMB=", None if row["peak_cuda_memory_mb"] is None else round(row["peak_cuda_memory_mb"], 2),
    )

print("\nREADY:", OUT)
print("NO TRAINING WAS RUN.")
